In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# 1. Recriação do DataFrame de features (para o notebook ser independente)
dados_ficticios = {
    'danceability': [0.8, 0.5, 0.6, 0.7, 0.4],
    'energy': [0.9, 0.4, 0.7, 0.6, 0.5],
    'loudness': [-5.0, -12.0, -8.0, -6.5, -10.0],
    'mode': [1, 0, 1, 1, 0],
    'speechiness': [0.05, 0.03, 0.08, 0.04, 0.06],
    'acousticness': [0.1, 0.8, 0.3, 0.2, 0.5],
    'instrumentalness': [0.0, 0.5, 0.01, 0.0, 0.02],
    'liveness': [0.2, 0.1, 0.15, 0.25, 0.18],
    'valence': [0.85, 0.3, 0.6, 0.7, 0.4],
    'tempo': [125.5, 90.0, 110.0, 118.0, 95.0]
}

df = pd.DataFrame(dados_ficticios)
audio_cols = list(df.columns)
scaler = MinMaxScaler()
df_features = df.copy()
df_features[audio_cols] = scaler.fit_transform(df[audio_cols])

# 2. Definição do Ambiente de Reforço
class MusicEnvironment:
    def __init__(self, df_features):
        self.df = df_features
        self.current_idx = 0
        
    def reset(self):
        self.current_idx = np.random.randint(0, len(self.df))
        return self.df.iloc[self.current_idx].values
        
    def step(self, next_idx):
        current_vector = self.df.iloc[self.current_idx].values.reshape(1, -1)
        next_vector = self.df.iloc[next_idx].values.reshape(1, -1)
        similarity = cosine_similarity(current_vector, next_vector)[0][0]
        
        if similarity < 0.4:
            reward = -1.0
            done = True
        elif 0.6 <= similarity <= 0.95:
            reward = 1.0
            done = False
        else:
            reward = 0.1
            done = False
            
        self.current_idx = next_idx
        next_state = self.df.iloc[self.current_idx].values
        return next_state, reward, done

# 3. Arquitetura da Rede Neural (DQN)
class DQNAgent(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQNAgent, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

state_size = len(df_features.columns)
action_size = len(df_features)

model = DQNAgent(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 4. Loop de Treinamento do Agente (Q-Learning)
env = MusicEnvironment(df_features)
episodes = 50
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.95

print("Iniciando o treinamento do agente...")

for episode in range(episodes):
    state = env.reset()
    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    total_reward = 0
    done = False
    
    while not done:
        if random.random() < epsilon:
            next_action = random.randint(0, action_size - 1)
        else:
            with torch.no_grad():
                q_values = model(state_tensor)
                next_action = torch.argmax(q_values).item()
        
        next_state, reward, done = env.step(next_action)
        total_reward += reward
        
        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0)
        
        with torch.no_grad():
            next_q_values = model(next_state_tensor)
            target = reward + gamma * torch.max(next_q_values).item()
            
        current_q = model(state_tensor)
        target_q = current_q.clone()
        target_q[0, next_action] = target
        
        loss = criterion(current_q, target_q)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        state_tensor = next_state_tensor
        
    if epsilon > 0.05:
        epsilon *= epsilon_decay
        
    print(f"Episódio {episode + 1}/{episodes} | Recompensa Total: {total_reward:.2f} | Epsilon: {epsilon:.2f}")

print("\nTreinamento concluído com sucesso!")

Iniciando o treinamento do agente...
Episódio 1/50 | Recompensa Total: 1.10 | Epsilon: 0.95
Episódio 2/50 | Recompensa Total: -1.00 | Epsilon: 0.90
Episódio 3/50 | Recompensa Total: -1.00 | Epsilon: 0.86
Episódio 4/50 | Recompensa Total: -0.70 | Epsilon: 0.81
Episódio 5/50 | Recompensa Total: 0.20 | Epsilon: 0.77
Episódio 6/50 | Recompensa Total: -1.00 | Epsilon: 0.74
Episódio 7/50 | Recompensa Total: -1.00 | Epsilon: 0.70
Episódio 8/50 | Recompensa Total: -1.00 | Epsilon: 0.66
Episódio 9/50 | Recompensa Total: 2.40 | Epsilon: 0.63
Episódio 10/50 | Recompensa Total: 4.30 | Epsilon: 0.60
Episódio 11/50 | Recompensa Total: 2.30 | Epsilon: 0.57
Episódio 12/50 | Recompensa Total: -0.90 | Epsilon: 0.54
Episódio 13/50 | Recompensa Total: 0.10 | Epsilon: 0.51
Episódio 14/50 | Recompensa Total: -1.00 | Epsilon: 0.49
Episódio 15/50 | Recompensa Total: -1.00 | Epsilon: 0.46
Episódio 16/50 | Recompensa Total: -1.00 | Epsilon: 0.44
Episódio 17/50 | Recompensa Total: 6.50 | Epsilon: 0.42
Episódio 1

C:\Users\LAISA\AppData\Local\Temp\ipykernel_5764\2874552264.py:91: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  state_tensor = torch.FloatTensor(state).unsqueeze(0)


Episódio 20/50 | Recompensa Total: 14.00 | Epsilon: 0.36
Episódio 21/50 | Recompensa Total: 1.00 | Epsilon: 0.34
Episódio 22/50 | Recompensa Total: 7.30 | Epsilon: 0.32
Episódio 23/50 | Recompensa Total: 10.60 | Epsilon: 0.31
Episódio 24/50 | Recompensa Total: 0.70 | Epsilon: 0.29
Episódio 25/50 | Recompensa Total: -1.00 | Epsilon: 0.28
Episódio 26/50 | Recompensa Total: -1.00 | Epsilon: 0.26
Episódio 27/50 | Recompensa Total: -1.00 | Epsilon: 0.25
Episódio 28/50 | Recompensa Total: -1.00 | Epsilon: 0.24
Episódio 29/50 | Recompensa Total: 2.10 | Epsilon: 0.23
Episódio 30/50 | Recompensa Total: -1.00 | Epsilon: 0.21
Episódio 31/50 | Recompensa Total: 11.40 | Epsilon: 0.20
Episódio 32/50 | Recompensa Total: 5.40 | Epsilon: 0.19
Episódio 33/50 | Recompensa Total: -0.90 | Epsilon: 0.18
Episódio 34/50 | Recompensa Total: 1.80 | Epsilon: 0.17
Episódio 35/50 | Recompensa Total: 1.80 | Epsilon: 0.17
Episódio 36/50 | Recompensa Total: -1.00 | Epsilon: 0.16
Episódio 37/50 | Recompensa Total: 3.0

In [2]:
# 5. Validação do Agente Treinado (Gerando uma Playlist Otimizada)
env = MusicEnvironment(df_features)
state = env.reset()
playlist_indices = [env.current_idx]

print("Gerando playlist otimizada com o agente DQN...")
for passo in range(4):  # Simula uma sequência de transições
    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    
    with torch.no_grad():
        q_values = model(state_tensor)
        next_action = torch.argmax(q_values).item() # Escolhe a melhor ação aprendida
    
    state, reward, done = env.step(next_action)
    playlist_indices.append(next_action)
    print(f"Passo {passo + 1} -> Próxima Música (Índice): {next_action} | Recompensa: {reward} | Skip? {done}")

print(f"\nSequência final de índices da playlist gerada: {playlist_indices}")

Gerando playlist otimizada com o agente DQN...
Passo 1 -> Próxima Música (Índice): 0 | Recompensa: 1.0 | Skip? False
Passo 2 -> Próxima Música (Índice): 0 | Recompensa: 0.1 | Skip? False
Passo 3 -> Próxima Música (Índice): 0 | Recompensa: 0.1 | Skip? False
Passo 4 -> Próxima Música (Índice): 0 | Recompensa: 0.1 | Skip? False

Sequência final de índices da playlist gerada: [3, 0, 0, 0, 0]


In [3]:
# 6. Salvamento do Modelo Treinado
torch.save(model.state_dict(), "vibe_bridge_dqn.pth")
print("Modelo DQN salvo com sucesso como 'vibe_bridge_dqn.pth'!")

Modelo DQN salvo com sucesso como 'vibe_bridge_dqn.pth'!
